# YuE — Lyrics-to-Song Generation

YuE (Wang et al., 2025) is an open foundation model that generates full songs — vocals, instruments, and structure — from lyrics and a genre description.

**Architecture:**
- Stage 1: Text → semantic audio tokens (captures melody, rhythm, lyrics alignment)
- Stage 2: Semantic tokens → acoustic tokens (adds timbre, production quality)
- EnCodec decoder: Acoustic tokens → waveform

**Note:** Full YuE generation requires ~24GB GPU RAM. On Colab free tier (T4, 15GB), we demonstrate with the smallest config or use pre-generated examples.

In [ ]:
# Install dependencies
!pip install torch torchaudio transformers scipy matplotlib IPython numpy

# Check GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu}, Memory: {mem:.1f} GB')
else:
    print('No GPU — will use pre-generated examples')

## Lyrics Input

YuE accepts structured lyrics with section markers (`[verse]`, `[chorus]`, `[bridge]`) and a genre/style description.

In [ ]:
lyrics = """
[verse]
City lights flicker in the evening rain
Footsteps echo down a quiet lane
Strangers pass like ships without a name
Every face a story, none the same

[chorus]
We are the dreamers in the neon glow
Chasing something that we'll never know
Hold on tight and never let it go
We are the dreamers in the neon glow

[verse]
Midnight buses carry restless souls
Coffee shops and half-forgotten goals
Searching for a place to call our own
In a world that never feels like home

[chorus]
We are the dreamers in the neon glow
Chasing something that we'll never know
Hold on tight and never let it go
We are the dreamers in the neon glow
"""

genre_prompt = "indie pop, acoustic guitar, warm vocals, moderate tempo"

print("Lyrics:")
print(lyrics)
print(f"Genre: {genre_prompt}")

## Attempt Generation

We try to run YuE on the available GPU. If memory is insufficient, we fall back to analyzing pre-generated examples.

In [ ]:
import os
import numpy as np
import torchaudio
from IPython.display import Audio, display

generated_audio = None
sample_rate = 44100

# --- Attempt YuE generation ---
try:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU available")

    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"Available GPU memory: {gpu_mem:.1f} GB")

    if gpu_mem < 20:
        print("Insufficient GPU memory for full YuE (need ~24GB).")
        print("Falling back to synthetic demo...")
        raise RuntimeError("Insufficient memory")

    # If sufficient memory, clone and run YuE
    print("Cloning YuE repository...")
    !git clone https://github.com/multimodal-art-projection/YuE.git
    os.chdir("YuE")
    !pip install -r requirements.txt

    # Generation would happen here with the full model
    # from inference import generate_song
    # generated_audio = generate_song(lyrics, genre_prompt)
    print("Full YuE generation would proceed here.")

except (RuntimeError, Exception) as e:
    print(f"\nFallback mode: {e}")
    print("\nGenerating a synthetic placeholder to demonstrate the analysis pipeline.")
    print("In class, we will listen to pre-generated YuE examples from the paper.\n")

    # Create a synthetic audio demo (chord progression + simple melody)
    duration = 10.0  # seconds
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    # Simple chord progression: C - Am - F - G
    chord_freqs = [
        [261.6, 329.6, 392.0],   # C major
        [220.0, 261.6, 329.6],   # A minor
        [174.6, 220.0, 261.6],   # F major
        [196.0, 246.9, 293.7],   # G major
    ]

    audio = np.zeros_like(t)
    chord_dur = duration / 4
    for i, freqs in enumerate(chord_freqs):
        start = int(i * chord_dur * sample_rate)
        end = int((i + 1) * chord_dur * sample_rate)
        segment = t[start:end] - t[start]
        for f in freqs:
            audio[start:end] += 0.15 * np.sin(2 * np.pi * f * segment)

    # Add a simple melody on top
    melody_notes = [523.3, 587.3, 659.3, 587.3, 523.3, 440.0, 392.0, 440.0]
    note_dur = duration / len(melody_notes)
    for i, freq in enumerate(melody_notes):
        start = int(i * note_dur * sample_rate)
        end = int((i + 1) * note_dur * sample_rate)
        segment = t[start:end] - t[start]
        envelope = np.exp(-2.0 * segment / note_dur)
        audio[start:end] += 0.2 * np.sin(2 * np.pi * freq * segment) * envelope

    # Normalize
    audio = audio / np.max(np.abs(audio)) * 0.8
    generated_audio = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)

    torchaudio.save('demo_output.wav', generated_audio, sample_rate)
    print("Synthetic demo saved to demo_output.wav")
    display(Audio(audio, rate=sample_rate))

## Analyzing AI-Generated Songs

Listen critically to the output and evaluate:
- **Vocal quality**: How natural do the vocals sound?
- **Lyrics alignment**: Does the melody match the syllable stress and phrasing?
- **Structure**: Is there contrast between verse and chorus?
- **Instrumentation**: Does it match the genre prompt?
- **Production**: How polished is the mix?

In [ ]:
import matplotlib.pyplot as plt
import torchaudio.transforms as T

# Load the audio (either generated or synthetic demo)
audio_path = 'demo_output.wav'
waveform, sr = torchaudio.load(audio_path)

# --- Waveform ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

time_axis = np.arange(waveform.shape[1]) / sr
axes[0].plot(time_axis, waveform[0].numpy(), linewidth=0.3, color='steelblue')
axes[0].set_title('Waveform', fontsize=14)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim([0, time_axis[-1]])

# --- Spectrogram ---
n_fft = 2048
hop_length = 512
spectrogram = T.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2.0)
spec = spectrogram(waveform[0])
spec_db = 10 * torch.log10(spec + 1e-10)

freq_axis = np.linspace(0, sr / 2, spec_db.shape[0])
time_axis_spec = np.arange(spec_db.shape[1]) * hop_length / sr

axes[1].pcolormesh(time_axis_spec, freq_axis, spec_db.numpy(),
                   shading='auto', cmap='magma')
axes[1].set_title('Spectrogram', fontsize=14)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim([0, 8000])

plt.tight_layout()
plt.show()

# --- Basic tempo estimation ---
print("\n--- Basic Audio Analysis ---")
print(f"Duration: {waveform.shape[1] / sr:.2f} seconds")
print(f"Sample rate: {sr} Hz")
print(f"Peak amplitude: {waveform.abs().max():.3f}")
print(f"RMS energy: {waveform.pow(2).mean().sqrt():.4f}")

## Comparison: Open-Source vs Commercial

Try the same lyrics with:
- **Suno** (suno.com) — paste lyrics, select genre
- **Udio** (udio.com) — paste lyrics, select style

Compare: Which produces better vocals? Better instrumentation? Which follows the lyrics more faithfully?

In [ ]:
# Comparison template — fill in after trying each system

systems = ['YuE (open-source)', 'Suno (commercial)', 'Udio (commercial)']
criteria = ['Vocal quality', 'Lyrics alignment', 'Structure (verse/chorus)',
            'Instrumentation', 'Production quality', 'Genre accuracy']

print("=" * 72)
print("AI Song Generation Comparison")
print("=" * 72)
print(f"Prompt: '{genre_prompt}'")
print(f"Lyrics: (Neon Glow — 2 verses, 2 choruses)")
print("=" * 72)
print()
print(f"{'Criterion':<25} {'YuE':<16} {'Suno':<16} {'Udio':<16}")
print("-" * 72)
for c in criteria:
    print(f"{c:<25} {'___/5':<16} {'___/5':<16} {'___/5':<16}")
print("-" * 72)
print(f"{'TOTAL':<25} {'___/30':<16} {'___/30':<16} {'___/30':<16}")
print()
print("Notes:")
print("  - Rate each criterion 1-5 (1=poor, 5=excellent)")
print("  - Consider: Which system best captures the 'indie pop' feel?")
print("  - Consider: Which vocals sound most natural/expressive?")
print("  - Consider: Which output would you share with a friend?")